## LLM Validation

In [ ]:
# pip install scikit-learn openpyxl httpx tqdm
# import sys
# !{sys.executable} -m pip install tabulate

# three repeated sequential runs on a random sample of 200 abstracts

In [3]:
import pandas as pd
from pathlib import Path

infile = Path("../table/gold_standard_30_march.xlsx")
outfile = Path("../table/gold_standard_random_200.xlsx")
outfile.parent.mkdir(parents=True, exist_ok=True)

df200 = pd.read_excel(infile, sheet_name="in").copy()

if "id" not in df200.columns:
    if "scopus_id" in df200.columns:
        df200["id"] = df200["scopus_id"].astype(str).str.strip()
    else:
        df200["id"] = df200.index.astype(str)

df200 = df200.sample(n=200, random_state=42).copy()
df200.to_excel(outfile, sheet_name="in", index=False)

print(f"Saved: {outfile}")
print(f"N = {len(df200)}")


Saved: ../table/gold_standard_random_200.xlsx
N = 200


In [53]:
import pandas as pd
from pathlib import Path
from sklearn.metrics import cohen_kappa_score

RUN_DIR = Path("../concordance/concordance_outputs")

run1_path = RUN_DIR / "run_rerun1_temp0.1_2026-04-02_222144.csv"
run2_path = RUN_DIR / "run_rerun2_temp0.1_2026-04-02_222354.csv"
run3_path = RUN_DIR / "run_rerun3_temp0.1_2026-04-02_222534.csv"

def load_run(path, label_name):
    df = pd.read_csv(path).copy()
    df[label_name] = df["llm_label"].astype(str).str.strip().str.upper().map({"YES": 1, "NO": 0})
    return df[[label_name]].reset_index(drop=True)

def compare_runs(df_a, col_a, df_b, col_b):
    paired = pd.concat([df_a, df_b], axis=1).dropna()
    kappa = cohen_kappa_score(paired[col_a], paired[col_b])
    agreement = (paired[col_a] == paired[col_b]).mean() * 100
    return {
        "comparison": f"{col_a} vs {col_b}",
        "N": len(paired),
        "kappa": round(kappa, 3),
        "agreement_pct": round(agreement, 1),
    }

run1 = load_run(run1_path, "run1")
run2 = load_run(run2_path, "run2")
run3 = load_run(run3_path, "run3")

results = pd.DataFrame([
    compare_runs(run1, "run1", run2, "run2"),
    compare_runs(run1, "run1", run3, "run3"),
    compare_runs(run2, "run2", run3, "run3"),
])

display(results)

for _, r in results.iterrows():
    print(f"{r['comparison']}: κ={r['kappa']:.3f}, N={r['N']}, agreement={r['agreement_pct']:.1f}%")


,comparison,N,kappa,agreement_pct
0,run1 vs run2,200,0.905,98.0
1,run1 vs run3,200,0.930,98.5
2,run2 vs run3,200,0.978,99.5


run1 vs run2: κ=0.905, N=200, agreement=98.0%
run1 vs run3: κ=0.930, N=200, agreement=98.5%
run2 vs run3: κ=0.978, N=200, agreement=99.5%


In [38]:
import pandas as pd
import numpy as np
from sklearn.metrics import cohen_kappa_score
from IPython.display import display, Markdown

# ----------------------------wo c
# SETTINGS
# ----------------------------
file_path = "../table/gold_standard_30_march.xlsx"
sheet_name = "in"

reference_cols = [
    "llm_policy_claim",
    ]

compare_cols = [
    "agreed_gold_standard",
    "agreed_gold_standard_with_exclusions",
    "DB review",
    "EC review",
    "MW review",
    "db re-review",
    "EC re-review",
]

# ----------------------------
# HELPER: recode binary values
# ----------------------------
na_strings = {"n/a", "na", "nan", "#n/a", "none", "", "-", "."}

def recode_binary(x):
    if pd.isna(x):
        return np.nan

    s = str(x).strip().lower()

    if s in na_strings:
        return np.nan

    yes_vals = {"1", "yes", "y", "true"}
    no_vals = {"0", "no", "n", "false"}

    if s in yes_vals:
        return 1
    if s in no_vals:
        return 0

    try:
        f = float(s)
        if f == 1:
            return 1
        if f == 0:
            return 0
    except:
        pass

    return np.nan

# ----------------------------
# LOAD DATA
# ----------------------------
df = pd.read_excel(file_path, sheet_name=sheet_name)

all_cols = reference_cols + compare_cols

missing_cols = []
for col in all_cols:
    if col not in df.columns:
        missing_cols.append(col)
    else:
        df[col + "_bin"] = df[col].apply(recode_binary)

def compare_to_reference(df, reference_col, other_col):
    ref_bin = reference_col + "_bin"
    oth_bin = other_col + "_bin"

    # Only rows where BOTH values are non-missing (0 or 1)
    sub = df[[ref_bin, oth_bin]].dropna().copy()
    sub = sub[sub[ref_bin].isin([0, 1]) & sub[oth_bin].isin([0, 1])]

    if len(sub) == 0:
        return {
            "comparison": f"{reference_col} vs {other_col}",
            "n": 0,
            "agreement_pct": np.nan,
            "kappa": np.nan,
            "sensitivity": np.nan,
            "specificity": np.nan,
        }

    sub[ref_bin] = sub[ref_bin].astype(int)
    sub[oth_bin] = sub[oth_bin].astype(int)

    agreement = (sub[ref_bin] == sub[oth_bin]).mean() * 100
    kappa = cohen_kappa_score(sub[ref_bin], sub[oth_bin])

    # Treat reference_col as truth
    tp = ((sub[ref_bin] == 1) & (sub[oth_bin] == 1)).sum()
    tn = ((sub[ref_bin] == 0) & (sub[oth_bin] == 0)).sum()
    fp = ((sub[ref_bin] == 0) & (sub[oth_bin] == 1)).sum()
    fn = ((sub[ref_bin] == 1) & (sub[oth_bin] == 0)).sum()

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan

    return {
        "comparison": f"{reference_col} vs {other_col}",
        "n": len(sub),
        "agreement_pct": round(agreement, 1),
        "kappa": round(kappa, 3),
        "sensitivity": round(sensitivity, 3) if pd.notna(sensitivity) else np.nan,
        "specificity": round(specificity, 3) if pd.notna(specificity) else np.nan,
    }


# ----------------------------
# RUN COMPARISONS
# ----------------------------
md = ""

if missing_cols:
    md += "**Missing columns:**\n"
    for col in missing_cols:
        md += f"- `{col}`\n"
    md += "\n"

for ref_col in reference_cols:
    if ref_col in missing_cols:
        continue

    results = []
    for col in compare_cols:
        if col not in missing_cols and col in df.columns:
            results.append(compare_to_reference(df, ref_col, col))
    if results:
        results_df = pd.DataFrame(results)
        md += results_df.to_markdown(index=False)
    else:
        md += "_No valid comparisons available._"
    md += "\n\n"

display(Markdown(md))


| comparison                                               |   n |   agreement_pct |   kappa |   sensitivity |   specificity |
|:---------------------------------------------------------|----:|----------------:|--------:|--------------:|--------------:|
| llm_policy_claim vs agreed_gold_standard                 | 204 |            92.6 |   0.803 |         0.768 |         0.986 |
| llm_policy_claim vs agreed_gold_standard_with_exclusions | 197 |            92.9 |   0.805 |         0.769 |         0.986 |
| llm_policy_claim vs DB review                            | 204 |            91.2 |   0.765 |         0.75  |         0.973 |
| llm_policy_claim vs EC review                            |  94 |            89.4 |   0.705 |         0.68  |         0.971 |
| llm_policy_claim vs MW review                            | 104 |            95.2 |   0.882 |         0.9   |         0.973 |
| llm_policy_claim vs db re-review                         |   5 |            80   |   0.545 |         1     |         0.5   |
| llm_policy_claim vs EC re-review                         | 204 |            92.6 |   0.803 |         0.768 |         0.986 |



In [39]:
sub = df[["DB review_bin", "EC review_bin"]].dropna().copy()
sub = sub[sub["DB review_bin"].isin([0, 1]) & sub["EC review_bin"].isin([0, 1])]

agreement = (sub["DB review_bin"] == sub["EC review_bin"]).mean() * 100
kappa = cohen_kappa_score(sub["DB review_bin"], sub["EC review_bin"])

print(f"DB vs EC: κ={kappa:.3f}, N={len(sub)}, agreement={agreement:.1f}%")

DB vs EC: κ=0.825, N=94, agreement=94.7%


In [44]:
import pandas as pd
import numpy as np
from pathlib import Path

review_path = Path("../table/supp_stratified_sample_400_blinded_db.xlsx")
main_path = Path("../data/analysis/analysis_dataset_enriched.csv")

review_df = pd.read_excel(review_path).copy()
main_df = pd.read_csv(main_path).copy()

main_df["scopus_id"] = main_df["scopus_id"].astype(str).str.strip()
main_df["doi"] = main_df["doi"].astype(str).str.strip().str.lower()

review_df["scopus_id"] = review_df["scopus_id"].astype(str).str.strip()
review_df["doi"] = review_df["doi"].astype(str).str.strip().str.lower()

In [45]:
print("main_df rows:", len(main_df))
print(main_df["publication_year"].min(), main_df["publication_year"].max())

main_df rows: 45807
1990 2024


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------------
# 1. Load files
# ------------------------------------------------------------------
review_path = Path("../table/supp_stratified_sample_400_blinded_db.xlsx")
main_path = Path("../data/analysis/analysis_dataset_enriched.csv")

review_df = pd.read_excel(review_path).copy()
main_df = pd.read_csv(main_path).copy()

# ------------------------------------------------------------------
# 2. Normalise keys / labels
# ------------------------------------------------------------------
def norm_text(x):
    if pd.isna(x):
        return pd.NA
    s = str(x).strip()
    return s if s else pd.NA

def norm_doi(x):
    if pd.isna(x):
        return pd.NA
    s = str(x).strip().lower()
    s = s.replace("https://doi.org/", "").replace("http://doi.org/", "")
    return s if s else pd.NA

def norm_bool(x):
    if pd.isna(x):
        return pd.NA
    if isinstance(x, bool):
        return x
    s = str(x).strip().lower()
    if s in {"1", "1.0", "true", "t", "yes", "y"}:
        return True
    if s in {"0", "0.0", "false", "f", "no", "n"}:
        return False
    return pd.NA

main_df["scopus_id"] = main_df["scopus_id"].apply(norm_text)
main_df["doi"] = main_df["doi"].apply(norm_doi)

review_df["scopus_id"] = review_df["scopus_id"].apply(norm_text)
review_df["doi"] = review_df["doi"].apply(norm_doi)

for col in ["DB_policy_claim", "EC_policy_claim", "db_double_checked"]:
    if col in review_df.columns:
        review_df[col] = review_df[col].apply(norm_bool)

main_df["llm_policy_claim"] = main_df["llm_policy_claim"].apply(norm_bool)
main_df["publication_year"] = pd.to_numeric(main_df["publication_year"], errors="coerce").astype("Int64")

print("Review rows:", len(review_df))
print("Main analytic rows:", len(main_df))

# ------------------------------------------------------------------
# 3. Build lookup tables
# ------------------------------------------------------------------
lookup_scopus = (
    main_df[["scopus_id", "publication_year", "llm_policy_claim"]]
    .dropna(subset=["scopus_id"])
    .drop_duplicates(subset=["scopus_id"])
    .copy()
)

lookup_doi = (
    main_df[["doi", "publication_year", "llm_policy_claim"]]
    .dropna(subset=["doi"])
    .drop_duplicates(subset=["doi"])
    .copy()
)

# ------------------------------------------------------------------
# 4. Match back to main dataset
# ------------------------------------------------------------------
review_merged = review_df.merge(
    lookup_scopus,
    on="scopus_id",
    how="left",
    validate="many_to_one"
)
review_merged["matched_by"] = np.where(review_merged["publication_year"].notna(), "scopus_id", pd.NA)

missing_mask = review_merged["publication_year"].isna()

if missing_mask.any():
    doi_fill = review_merged.loc[missing_mask, ["doi"]].merge(
        lookup_doi,
        on="doi",
        how="left",
        validate="many_to_one"
    )
    review_merged.loc[missing_mask, "publication_year"] = doi_fill["publication_year"].values
    review_merged.loc[missing_mask, "llm_policy_claim"] = doi_fill["llm_policy_claim"].values
    review_merged.loc[
        missing_mask & doi_fill["publication_year"].notna().values, "matched_by"
    ] = "doi"

# ------------------------------------------------------------------
# 5. Final manual label
# ------------------------------------------------------------------
# If DB and EC agree, use that.
# If DB and EC disagree, use db_double_checked as adjudication.
review_merged["manual_claim_final"] = np.where(
    review_merged["DB_policy_claim"] == review_merged["EC_policy_claim"],
    review_merged["DB_policy_claim"],
    review_merged["db_double_checked"]
)
review_merged["manual_claim_final"] = pd.Series(review_merged["manual_claim_final"]).astype("boolean")

# ------------------------------------------------------------------
# 6. Checks
# ------------------------------------------------------------------
n_total = len(review_merged)
n_matched = int(review_merged["publication_year"].notna().sum())
n_unmatched = int(review_merged["publication_year"].isna().sum())
n_discordant = int((review_merged["DB_policy_claim"] != review_merged["EC_policy_claim"]).fillna(False).sum())
n_final_nonmissing = int(review_merged["manual_claim_final"].notna().sum())

print(f"\nManual review file rows: {n_total}")
print(f"Matched back to analytic sample: {n_matched}")
print(f"Unmatched rows: {n_unmatched}")
print(f"DB/EC discordant rows: {n_discordant}")
print(f"Rows with final manual label: {n_final_nonmissing}")

print("\nMatch method counts:")
print(review_merged["matched_by"].value_counts(dropna=False))

if n_unmatched:
    print("\nUnmatched rows preview:")
    display(review_merged.loc[
        review_merged["publication_year"].isna(),
        ["review_id", "scopus_id", "doi", "title"]
    ].head(20))

# ------------------------------------------------------------------
# 7. Overall summary
# ------------------------------------------------------------------
matched_review = review_merged[review_merged["publication_year"].notna()].copy()

n_manual_claim = int(matched_review["manual_claim_final"].fillna(False).sum())
manual_claim_rate = 100 * n_manual_claim / len(matched_review) if len(matched_review) else np.nan

n_llm_claim = int(matched_review["llm_policy_claim"].fillna(False).sum())
llm_claim_rate = 100 * n_llm_claim / len(matched_review) if len(matched_review) else np.nan

overall_summary = pd.DataFrame([{
    "n_reviewed": n_total,
    "n_matched_to_main_df": len(matched_review),
    "n_db_ec_discordant": n_discordant,
    "n_manual_policy_claim": n_manual_claim,
    "manual_policy_claim_rate_pct": round(manual_claim_rate, 1) if pd.notna(manual_claim_rate) else np.nan,
    "n_llm_policy_claim": n_llm_claim,
    "llm_policy_claim_rate_pct": round(llm_claim_rate, 1) if pd.notna(llm_claim_rate) else np.nan,
}])

display(overall_summary)

# ------------------------------------------------------------------
# 8. Period summaries
# ------------------------------------------------------------------
periods = {
    "1990-1999": (1990, 2000),
    "2000-2009": (2000, 2010),
    "2010-2019": (2010, 2020),
    "2020-2024": (2020, 2025),
}

rows = []
for label, (start, end) in periods.items():
    full_sub = main_df[
        (main_df["publication_year"] >= start) &
        (main_df["publication_year"] < end)
    ].copy()

    review_sub = matched_review[
        (matched_review["publication_year"] >= start) &
        (matched_review["publication_year"] < end)
    ].copy()

    n_full = len(full_sub)
    n_review = len(review_sub)

    full_rate = 100 * full_sub["llm_policy_claim"].astype(bool).mean() if n_full else np.nan
    sample_llm_rate = 100 * review_sub["llm_policy_claim"].astype(bool).mean() if n_review else np.nan
    manual_rate = 100 * review_sub["manual_claim_final"].astype(bool).mean() if n_review else np.nan

    rows.append({
        "period": label,
        "n_full": n_full,
        "full_llm_claim_rate_pct": round(full_rate, 1) if pd.notna(full_rate) else np.nan,
        "n_review_sample": n_review,
        "sample_llm_claim_rate_pct": round(sample_llm_rate, 1) if pd.notna(sample_llm_rate) else np.nan,
        "manual_claim_rate_pct": round(manual_rate, 1) if pd.notna(manual_rate) else np.nan,
    })

period_summary = pd.DataFrame(rows)
display(period_summary)

Review rows: 400
Main analytic rows: 45807

Manual review file rows: 400
Matched back to analytic sample: 400
Unmatched rows: 0
DB/EC discordant rows: 24
Rows with final manual label: 400

Match method counts:
matched_by
scopus_id    400
Name: count, dtype: int64


,n_reviewed,n_matched_to_main_df,n_db_ec_discordant,n_manual_policy_claim,manual_policy_claim_rate_pct,n_llm_policy_claim,llm_policy_claim_rate_pct
0,400,400,24,127,31.8,97,24.2


,period,n_full,full_llm_claim_rate_pct,n_review_sample,sample_llm_claim_rate_pct,manual_claim_rate_pct
0,1990-1999,10436,17.6,114,13.2,19.3
1,2000-2009,12529,22.8,115,22.6,27.8
2,2010-2019,15464,28.4,114,27.2,36.0
3,2020-2024,7378,35.8,57,43.9,56.1
